# Batch processing

Run your saved planner on all 100 public competition scenarios. Each task runs an
interactive SUMO simulation, then validates and scores the executed drive. The
batch exports completed drives and records failures while continuing to the next
task. Attempting every scenario does not guarantee that your planner solves it.

First select your planner in
[my_submission/student_planner.py](../my_submission/student_planner.py), as explained
in [tutorial 04](04_reuse_and_evaluate_planners.ipynb). Save all source files and
keep them unchanged during the batch. Start this notebook with the competition
Python environment and a fresh kernel.


## Configure the batch

`num_workers = 1` processes scenarios sequentially and lets you debug in the kernel.
For parallel processing, choose a larger number that fits your available CPU cores
and memory, for example `2` or `6`. Each worker uses a separate process because SUMO
and the submission loader keep process-wide state. Numeric libraries use one thread
per worker.

All runs use the competition settings: SM1, seed 1234 and a maximum of 200 simulation
steps. Configure the planner itself in `create_planner()`. The step allowance is
simulated time, not a wall-clock timeout.


In [ ]:
import os
from pathlib import Path

# Set these before importing the planner or starting worker processes.
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

num_workers = 1
root = Path.cwd()
if root.name == "tutorials":
    root = root.parent


In [ ]:
import csv
import json
from collections import Counter
from dataclasses import asdict

from planner.common.scenarios import public_scenarios
from planner.runtime.batch import run_batch

submission = root / "my_submission"
solution_directory = root / "solutions"
report_directory = root / "outputs" / "batch_processing"
scenario_paths = public_scenarios()
# For a short first check, uncomment: scenario_paths = scenario_paths[:3]
print(f"Selected {len(scenario_paths)} scenarios; workers={num_workers}")


## Run the scenarios

Run this section once to start the selected batch. Parallel results appear in
completion order. Every completed drive is exported under
`solutions/<scenario_id>/` with `solution.xml`, `scenario.xml` and `metadata.json`
from that same simulation, including invalid or unscored drives for inspection.
A planning failure produces a report entry without a new result pair.

Rerunning replaces the reports and each newly completed scenario's three result
files. If a failed rerun leaves an older result, its report entry has
`previous_result_retained=True`; those files are not a result of the new attempt.
Use another `solution_directory` and `report_directory` to keep separate experiments.


In [ ]:
report_directory.mkdir(parents=True, exist_ok=True)
progress_path = report_directory / "results.json"
report_csv = report_directory / "results.csv"
report_csv.unlink(missing_ok=True)  # Clear the previous run's summary.
results = []

def save_progress():
    temporary = progress_path.with_suffix(".tmp")
    temporary.write_text(json.dumps([asdict(item) for item in results], indent=2) + "\n")
    temporary.replace(progress_path)

save_progress()


In [ ]:
for record in run_batch(
    scenario_paths, submission, solution_directory, workers=num_workers
):
    results.append(record)
    save_progress()
    cost = "—" if record.cost is None else f"{record.cost:.3f}"
    elapsed = "—" if record.elapsed_seconds is None else f"{record.elapsed_seconds:.2f}"
    print(
        f"[{len(results)}/{len(scenario_paths)}] {record.scenario_name}: "
        f"{record.status}, SM1={cost}, seconds={elapsed}",
        flush=True,
    )
    if record.status != "passed":
        print(f"  {record.message}", flush=True)
    if record.previous_result_retained:
        if record.status == "worker_error":
            print("  Result metadata exists; inspect the files because the worker's completion is unknown.", flush=True)
        else:
            print("  Previous result retained; this attempt did not export a new pair.", flush=True)


## Review and save the summary

`passed` means the drive is valid, has an SM1 score and was exported successfully.
`invalid_solution` means final validation rejected the completed drive.
`scoring_error` means a valid drive could not receive a score, and `export_error`
means the output files could not be saved. Other statuses describe input or planner
failures, such as `no_path`, `incompatible_state` or `invalid_output`. A
`worker_error` means a spawned process failed; its task may not have started.
Elapsed time is unavailable for that task. If result metadata exists after a
worker error, inspect the files: the worker may have failed after exporting.

The JSON report is updated after every finished task. The next cells summarize the
records received so far and write a CSV table, with one row per attempt. SM1 is a
per-scenario cost; lower is better. A local batch cannot determine competition
first places without the other participants' results.


In [ ]:
summary = {
    "selected": len(scenario_paths),
    "finished": len(results),
    "valid_drives": sum(item.valid is True for item in results),
    "scored_and_exported": sum(item.status == "passed" for item in results),
    "statuses": dict(Counter(item.status for item in results)),
}
summary


In [ ]:
rows = [asdict(item) for item in sorted(results, key=lambda item: item.scenario_name)]
report_csv = report_directory / "results.csv"
if rows:
    with report_csv.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=rows[0])
        writer.writeheader()
        writer.writerows(rows)
print(progress_path)
print(report_csv)


In [ ]:
# Inspect the messages and retained-result flags for unsuccessful attempts.
[item for item in rows if item["status"] != "passed"]


Use [tutorial 03](03_solution_and_cost_validation.ipynb) to reload, validate and
score an exported ego/traffic pair. Review the report before choosing which result
folders to submit, especially when a failed attempt retained older files. Package
the exact generating source and dependency files using
[tutorial 04](04_reuse_and_evaluate_planners.ipynb#Package-and-submit).

The equivalent sequential command from the repository root is:

```sh
python -m planner --public --submission my_submission --solutions solutions
```
